# Inimene-tsüklis töövoog Microsoft Agent Framework'iga

## 🎯 Õpieesmärgid

Selles märkmikus õpid, kuidas rakendada **inimene-tsüklis** töövooge kasutades Microsoft Agent Framework'i `RequestInfoExecutor`-it. See võimas muster võimaldab peatada tehisintellekti töövood, et koguda inimeste sisendit, muutes agentid interaktiivseks ja andes inimestele kontrolli oluliste otsuste üle.

## 🔄 Mis on Inimene-tsüklis?

**Inimene-tsüklis (HITL)** on disainimuster, kus tehisintellekti agent peatab täitmise, et küsida inimestelt sisendit enne jätkamist. See on oluline:

- ✅ **Olulised otsused** - Saada inimkinnitust enne tähtsate toimingute tegemist
- ✅ **Kaheldavad olukorrad** - Lase inimestel selgitada, kui AI on ebakindel
- ✅ **Kasutajale eelistused** - Küsige kasutajatelt, et valida mitme valiku vahel
- ✅ **Järgimine ja ohutus** - Tagada inimeste järelevalve reguleeritud tegevuste puhul
- ✅ **Interaktiivsed kogemused** - Loo vestlusagentide, kes reageerivad kasutaja sisendile

## 🏗️ Kuidas see Microsoft Agent Framework'is töötab

Raamistik pakub HITL jaoks kolme peamist komponenti:

1. **`RequestInfoExecutor`** - Eriline täitja, mis peatab töövoo ja saadab `RequestInfoEvent`
2. **`RequestInfoMessage`** - Baasklass tüübitud päringukuval olevate inimeste sõnumite jaoks
3. **`RequestResponse`** - Seob inimeste vastused algsete päringutega kasutades `request_id`

**Töövoo muster:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Meie näide: Hotelli broneerimine kasutaja kinnitusega

Laiendame tingimuslikku töövoogu, lisades inimese kinnituse **enne** alternatiivsete sihtkohtade pakkumist:

1. Kasutaja palub sihtkohta (nt "Pariis")
2. `availability_agent` kontrollib tubade saadavust
3. **Kui tube pole** → `confirmation_agent` küsib "Kas soovite näha alternatiive?"
4. Töövoog **peatub** `RequestInfoExecutor` abil
5. **Inimene vastab** "jah" või "ei" konsoolisisendi kaudu
6. `decision_manager` juhib vastuste põhjal:
   - **Jah** → Näita alternatiivseid sihtkohti
   - **Ei** → Tühista broneeringu taotlus
7. Kuvatakse lõpptulemus

See näitab, kuidas anda kasutajatele kontroll agentide soovituste üle!

---

Alustame! 🚀


## Samm 1: Vajalike teekide importimine

Impordime tavapärased Agent Framework komponendid ning **inimese sekkumist nõudvad klassid**:
- `RequestInfoExecutor` - Täitja, mis peatab töövoo inimese sisendi jaoks
- `RequestInfoEvent` - Sündmus, mis käivitub inimese sisendi nõudmisel
- `RequestInfoMessage` - Aluskklass tüübistatud päringute jaoks
- `RequestResponse` - Seob inimese vastused päringutega
- `WorkflowOutputEvent` - Sündmus töövoo väljundite tuvastamiseks


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 2. samm: määratle Pydantic mudelid struktureeritud väljundite jaoks

Need mudelid määratlevad **skeemi**, mida agendid tagastavad. Hoidame kõiki mudelid tingimuslikust töövoost ja lisame:

**Uus inimese kaasamise jaoks:**
- `HumanFeedbackRequest` - `RequestInfoMessage` alamklass, mis määratleb inimesele saadetava päringu koormuse
  - Sisaldab `prompt` (küsimust) ja `destination` (konteksti kättesaamatu linna kohta)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 3. samm: Loo hotelli broneerimise tööriist

Sama tööriist nagu tingimuslikus töös voog - kontrollib, kas sihtkohas on toad saadaval.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4. samm: defineeri marsruutimiseks tingimusfunktsioonid

Meil on vaja **neli tingimusfunktsiooni** meie inim-tsükliga töövoo jaoks:

**Olemasolevast tingimuslikust töövoost:**
1. `has_availability_condition` - Marsruudib, kui hotellid ON saadaval
2. `no_availability_condition` - Marsruudib, kui hotellid EI OLE saadaval

**Uus inim-tsükliga töövoo jaoks:**
3. `user_wants_alternatives_condition` - Marsruudib, kui kasutaja ütleb "jah" alternatiivide kohta
4. `user_declines_alternatives_condition` - Marsruudib, kui kasutaja ütleb "ei" alternatiivide kohta


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 5. samm: Loo otsuse halduri täitja

See on **inimene-tsüklis mustri tuumik**! `DecisionManager` on kohandatud `Executor`, mis:

1. **Võtab vastu inimtagasisidet** `RequestResponse` objektide kaudu
2. **Töötab läbi kasutaja otsuse** (jah/ei)
3. **Suunab töölõiku** saates sõnumeid sobivatele agentidele

Põhifunktsioonid:
- Kasutab meetodite töötlusetappidena avalikustamiseks `@handler` dekoratiivset
- Võtab vastu `RequestResponse[HumanFeedbackRequest, str]`, mis sisaldab nii algset päringut kui kasutaja vastust
- Tagastab lihtsaid "jah" või "ei" sõnumeid, mis käivitavad meie tingimusfunktsioonid


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## 6. samm: loo kohandatud kuvamise täitja

Sama kuvamise täitja tingimuslikust töövoost - annab töövoo väljundina lõplikud tulemused.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Samm 7: Laadi keskkonnamuutujad

Konfigureeri LLM klient (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Samm 8: Loo AI agentide ja täitjate

Loome **kuus töövoo komponenti**:

**Agendid (mähitud AgentExecutorisse):**
1. **availability_agent** - Kontrollib hotelli saadavust tööriista abil
2. **confirmation_agent** - 🆕 Valmistab ette inimkinnituse taotluse
3. **alternative_agent** - Pakub alternatiivseid linnu (kui kasutaja ütleb jah)
4. **booking_agent** - Julgustab broneerima (kui toad on saadaval)
5. **cancellation_agent** - 🆕 Käitleb tühistusteadet (kui kasutaja ütleb ei)

**Spetsiaalsed täitjad:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, mis peatab töövoo inimsisendi ootel
7. **decision_manager** - 🆕 Kohandatud täitja, mis juhib vastavalt inimese vastusele (muide juba eelnevalt defineeritud)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 9. samm: ehita töövoog koos inimese sekkumisega

Nüüd ehitame töövoo graafi koos **tingimusliku marsruutimisega**, mis hõlmab inimese sekkumise rada:

**Töövoo struktuur:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Põhijooned:**
- `availability_agent → confirmation_agent` (kui ruume pole)
- `confirmation_agent → prepare_human_request` (tüübi teisendus)
- `prepare_human_request → request_info_executor` (pause inimese jaoks)
- `request_info_executor → decision_manager` (alati - tagastab RequestResponse)
- `decision_manager → alternative_agent` (kui kasutaja ütleb "jah")
- `decision_manager → cancellation_agent` (kui kasutaja ütleb "ei")
- `availability_agent → booking_agent` (kui ruumid on saadaval)
- Kõik rajad lõppevad `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Samm 10: Käivita Testjuhtum 1 - Linn ILMA Saadavuseta (Pariis koos inimkinnitusega)

See test demonstreerib **täielikku inimtsüklit**:

1. Taotlus hotellile Pariisis
2. availability_agent kontrollib → Tube pole
3. confirmation_agent loob inimesele suunatud küsimuse
4. request_info_executor **peatab töövoo** ja väljastab `RequestInfoEvent`
5. **Rakendus tuvastab sündmuse ja pakub kasutajale konsoolis valikut**
6. Kasutaja sisestab "jah" või "ei"
7. Rakendus saadab vastuse läbi `send_responses_streaming()`
8. decision_manager suunab vastuse põhjal
9. Kuvatakse lõplik tulemus

**Oluline muster:**
- Kasuta esimese iteratsiooni puhul `workflow.run_stream()`
- Kasuta järgnevatel iteratsioonidel `workflow.send_responses_streaming(pending_responses)`
- Kuula `RequestInfoEvent` sündmust, et tuvastada vajadus inimese sisendi järele
- Kuula `WorkflowOutputEvent` sündmust, et saada lõplikud tulemused


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Samm 11: Käivita Test Case 2 - Linn SAADAVOLJUSEGA (Stockholm - inimeste sisendit ei ole vaja)

See test näitab **otse rada**, kui toad on saadaval:

1. Palu hotelli Stockholmis
2. availability_agent kontrollib → toad saadaval ✅
3. booking_agent pakub broneerimist
4. display_result näitab kinnitust
5. **Inimeste sisendit ei nõuta!**

Töövoog möödub inimeste sekkumisest täielikult, kui toad on saadaval.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Peamised punktid ja inimlõikus parimad tavad

### ✅ Mida sa õppisid:

#### 1. **RequestInfoExecutor muster**
Microsoft Agent Frameworkis kasutatav inimlõikus muster kasutab kolme põhikomponenti:
- `RequestInfoExecutor` - Peatab töövoo ja käivitab sündmusi
- `RequestInfoMessage` - Tüübistatavate päringuandmete baasklass (alimklassiks!)
- `RequestResponse` - Seob inimvastused algsete päringutega

**Oluline mõistmine:**
- `RequestInfoExecutor` ei kogu ise sisendit - see ainult peatab töövoo
- Sinu rakenduse kood peab kuulama `RequestInfoEvent` ja koguma sisendi
- Sa pead kutsuma `send_responses_streaming()` koos sõnastikuga, mis seob `request_id` kasutaja vastusega

#### 2. **Streaming käivitamise muster**
```python
# Esimene kordus
stream = workflow.run_stream(initial_request)

# Järgmised kordused (pärast inimese sisendit)
stream = workflow.send_responses_streaming(pending_responses)

# Töötle sündmusi alati
events = [event async for event in stream]
```

#### 3. **Sündmustel põhinev arhitektuur**
Kuula konkreetseid sündmusi töövoo juhtimiseks:
- `RequestInfoEvent` - Vaja on inimsisendit (töövoog peatatud)
- `WorkflowOutputEvent` - Lõplik tulemus on saadaval (töövoog lõpetatud)
- `WorkflowStatusEvent` - Oleku muutused (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS jms)

#### 4. **Kohandatud täitjad koos @handler'iga**
`DecisionManager` näitab, kuidas luua täitjaid, kes:
- Kasutavad `@handler` dekoratiivmärgendit meetodite eksponeerimiseks töövoo sammudena
- Võtavad vastu tüübistatavaid sõnumeid (nt `RequestResponse[HumanFeedbackRequest, str]`)
- Juhtivad töövoogu sõnumite saatmisega teistele täitjatele
- Saab juurde pääseda kontekstile läbi `WorkflowContext`

#### 5. **Tingimuslik marsruutimine inimotsustega**
Sa saad luua tingimusfunktsioone, mis hindavad inimvastuseid:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Reaalse maailma rakendused:

1. **Kinnitustöövood**
   - Saada juhilt kinnitust enne kuludeklaratsioonide töötlemist
   - Nõua inimlikku ülevaatust enne automaatsete meilide saatmist
   - Kinnita kõrge väärtusega tehingud enne täitmist

2. **Sisu modereerimine**
   - Märgi kahtlust äratav sisu inimliku ülevaatuse jaoks
   - Palu moderaatoritel teha lõplik otsus piirjuhtumite puhul
   - Suuna juhtumid inimestele, kui tehisintellekti kindlus on madal

3. **Klienditeenindus**
   - Lase tehisintellektil automaatselt hallata rutiinseid küsimusi
   - Suuna keerulised juhtumid inimagentidele
   - Küsi kliendilt, kas ta soovib rääkida inimesega

4. **Andmetöötlus**
   - Palu inimestel lahendada mitmeti tõlgendatavad andmekirjed
   - Kinnita AI tõlgendused ebaselgetest dokumentidest
   - Lase kasutajatel valida mitme sobiva tõlgenduse vahel

5. **Ohutusega seotud süsteemid**
   - Nõua inimkinnitust enne pöördumatuid toiminguid
   - Saada luba enne tundlike andmete ligipääsu
   - Kinnita otsused reguleeritud tööstusharudes (tervishoid, finants)

6. **Interaktiivsed agendid**
   - Loo vestlusbotid, kes esitavad täiendavaid küsimusi
   - Loo nõuandjad, kes juhendavad kasutajaid keerulistes protsessides
   - Kujunda agendid, kes teevad samm-sammult koostööd inimestega

### 🔄 Võrdlus: inimlõikusega vs ilma

| Omadus | Tingimuslik töövoog | Inimlõikusega töövoog |
|---------|---------------------|---------------------------|
| **Täideviimine** | Üksik `workflow.run()` | Tsükkel `run_stream()` + `send_responses_streaming()` |
| **Kasutajasisend** | Puudub (täielikult automatiseeritud) | Interaktiivsed küsitlused `input()` või kasutajaliidese kaudu |
| **Komponendid** | Agendid + Täitjad | + RequestInfoExecutor + DecisionManager |
| **Sündmused** | Ainult AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent jt |
| **Peatused** | Peatumist pole | Töövoog peatub RequestInfoExecutor'is |
| **Inimjuhtimine** | Inimjuhtimist pole | Inimesed teevad olulisi otsuseid |
| **Kasutusjuht** | Automatiseerimine | Koostöö ja järelevalve |

### 🚀 Täiustatud mustrid:

#### Mitmed inimotsuse punktid
Sama töövoo sees võib olla mitu `RequestInfoExecutor` sõlme:
```python
.add_edge(agent1, request_info_1)  # Esimene inimotsus
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Teine inimotsus
.add_edge(decision_manager_2, final_agent)
```

#### Aegumise haldus
Rakenda inimvastuste jaoks aegumisi:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Vaikimisi turvaline valik
```

#### Rikas kasutajaliidese integreerimine
`input()` asemel integreeru veebiliidese, Slacki, Teamsi jms-ga:
```python
if isinstance(event, RequestInfoEvent):
    # Saada teavitus kasutaja eelistatud kanalile
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Tingimuslik inimlõikus
Küsi inimvastuseid ainult spetsiifilistes olukordades:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Suuna inimeseni ainult siis, kui kindlustunne on madal või väärtus on kõrge
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Parimad tavad:

1. **Alamklassi RequestInfoMessage alati**
   - Tagab tüübisõltumatuse ja valideerimise
   - Võimaldab rikkalikku konteksti kasutajaliidese joonistamiseks
   - Selgitab iga päringutüübi kavatsust

2. **Kasuta kirjeldavaid kutsungeid**
   - Lisa kontekst, mida küsin
   - Selgita iga valiku tagajärgi
   - Hoia küsimused lihtsad ja arusaadavad

3. **Käsitle ootamatuid sisendeid**
   - Kontrolli kasutaja vastuseid
   - Paku vaikeväärtused vigastele sisenditele
   - Anna selged veateated

4. **Jälgi päringu IDsid**
   - Kasuta seost request_id ja vastuste vahel
   - Ära püüa olekut käsitsi juhtida

5. **Märgista mitteblokeerimiseks**
   - Ära blokeeri lugusid sisendi ootamiseks
   - Kasuta asünkroonseid mustreid üle kogu süsteemi
   - Toeta samaaegseid töövoo eksemplare

### 📚 Seotud mõisted:

- **Agent Middleware** - Intercepteerib agentide kõnesid (eelmine märkmik)
- **Töövoo oleku haldus** - Säilita töövoo olek käivituste vahel
- **Mitme-agendi koostöö** - Ühenda inimlõikus agenditiimidega
- **Sündmustel põhinevad arhitektuurid** - Ehita reageerivaid süsteeme sündmustega

---

### 🎓 Palju õnne!

Sa valdad nüüd Microsoft Agent Frameworki inimlõikusega töövooge! Sa tead nüüd, kuidas:
- ✅ Peatada töövooge inimsisendi kogumiseks
- ✅ Kasutada RequestInfoExecutorit ja RequestInfoMessage'i
- ✅ Käsitleda sündmustega voogesituse täitmist
- ✅ Luua kohandatud täitjaid koos @handler'iga
- ✅ Juhtida töövoogusid inimotsuste alusel
- ✅ Ehita interaktiivseid tehisintellekti agente koostööks inimestega

**See on oluline muster usaldusväärsete, kontrollitavate AI süsteemide ehitamiseks!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
